In [ ]:
import os 
os.chdir("/hpc/home/ephdh/workspace/suzhou_false_validation/analysis")

import ast
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score, confusion_matrix
from sklearn.utils import resample

In [ ]:
result_df = pd.read_csv(
        "/hpc/home/ephdh/workspace/suzhou_false_validation/data/meta_data/meta_data_with_model_scores.csv", 
        encoding="utf-16",  
        converters={"DICOMPaths": ast.literal_eval, 
                    'clip_image_scores': ast.literal_eval,
                    'cnn_image_scores': ast.literal_eval}
        )
result_df['DICOM_paths'] = result_df['DICOM_paths'].apply(ast.literal_eval)

In [ ]:
with pd.option_context('display.max_columns', None):
    display(result_df.head())

In [ ]:
# ==========================================
# 1. Configuration & Data Prep
# ==========================================
TARGET_THRESHOLD = 0.536

# --- Synthetic Data Generation (Replace with your actual data loading) ---
# df = pd.read_csv('your_file.csv') 
df = result_df.copy()
# -------------------------------------------------------------------------

# Helper: Bootstrap CI
def calculate_auc_ci(y_true, y_scores, n_bootstraps=1000, rng_seed=42):
    rng = np.random.RandomState(rng_seed)
    bootstrapped_scores = []
    y_true = np.array(y_true)
    y_scores = np.array(y_scores)
    
    for i in range(n_bootstraps):
        indices = rng.randint(0, len(y_scores), len(y_scores))
        if len(np.unique(y_true[indices])) < 2:
            continue
        score = roc_auc_score(y_true[indices], y_scores[indices])
        bootstrapped_scores.append(score)

    sorted_scores = np.array(bootstrapped_scores)
    sorted_scores.sort()
    lower = sorted_scores[int(0.025 * len(sorted_scores))]
    upper = sorted_scores[int(0.975 * len(sorted_scores))]
    return lower, upper

# Helper: Calculate Sens/Spec at Threshold
def get_metrics_at_threshold(y_true, y_scores, threshold):
    y_pred = (y_scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    fpr_point = 1 - specificity
    tpr_point = sensitivity
    
    return sensitivity, specificity, fpr_point, tpr_point

# ==========================================
# 2. Performance Calculation
# ==========================================

# --- A. Overall Cohort ---
y_true_all = df['GroundTruth']
y_score_all = df['ensemble_patient_score'].astype(float)
fpr_all, tpr_all, _ = roc_curve(y_true_all, y_score_all)
auc_all = auc(fpr_all, tpr_all)
ci_low_all, ci_high_all = calculate_auc_ci(y_true_all, y_score_all)
sens_all, spec_all, fpr_dot_all, tpr_dot_all = get_metrics_at_threshold(y_true_all, y_score_all, TARGET_THRESHOLD)

# --- B. Radiologically Correct (TP, TN) ---
mask_corr = df['Group'].isin(['TP', 'TN'])
y_true_corr = df.loc[mask_corr, 'GroundTruth']
y_score_corr = df.loc[mask_corr, 'ensemble_patient_score'].astype(float)
fpr_corr, tpr_corr, _ = roc_curve(y_true_corr, y_score_corr)
auc_corr = auc(fpr_corr, tpr_corr)
ci_low_corr, ci_high_corr = calculate_auc_ci(y_true_corr, y_score_corr)
sens_corr, spec_corr, fpr_dot_corr, tpr_dot_corr = get_metrics_at_threshold(y_true_corr, y_score_corr, TARGET_THRESHOLD)

# --- C. Radiologically Incorrect (FP, FN) ---
mask_inc = df['Group'].isin(['FP', 'FN'])
y_true_inc = df.loc[mask_inc, 'GroundTruth']
y_score_inc = df.loc[mask_inc, 'ensemble_patient_score'].astype(float)
fpr_inc, tpr_inc, _ = roc_curve(y_true_inc, y_score_inc)
auc_inc = auc(fpr_inc, tpr_inc)
ci_low_inc, ci_high_inc = calculate_auc_ci(y_true_inc, y_score_inc)
sens_inc, spec_inc, fpr_dot_inc, tpr_dot_inc = get_metrics_at_threshold(y_true_inc, y_score_inc, TARGET_THRESHOLD)

# ==========================================
# 3. Plotting
# ==========================================
# plt.figure(figsize=(10, 9))

# # Plot Curves
# plt.plot(fpr_all, tpr_all, color='navy', lw=3, zorder=3,
#          label=f'Overall\n(AUC={auc_all:.2f} [{ci_low_all:.2f}-{ci_high_all:.2f}])')

# plt.plot(fpr_corr, tpr_corr, color='forestgreen', lw=2, linestyle='--', zorder=2,
#          label=f'Rad Correct\n(AUC={auc_corr:.2f} [{ci_low_corr:.2f}-{ci_high_corr:.2f}])')

# plt.plot(fpr_inc, tpr_inc, color='firebrick', lw=2, linestyle='-.', zorder=2,
#          label=f'Rad Incorrect\n(AUC={auc_inc:.2f} [{ci_low_inc:.2f}-{ci_high_inc:.2f}])')

# # Plot Diagonal
# plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle=':', zorder=1)

# # Plot Operating Points
# plt.scatter(fpr_dot_all, tpr_dot_all, color='navy', s=100, zorder=5, edgecolor='white', marker='o')
# plt.scatter(fpr_dot_corr, tpr_dot_corr, color='forestgreen', s=100, zorder=5, edgecolor='white', marker='s')
# plt.scatter(fpr_dot_inc, tpr_dot_inc, color='firebrick', s=100, zorder=5, edgecolor='white', marker='^')

# # Formatting
# plt.xlim([0.0, 1.0])
# plt.ylim([0.0, 1.05])
# plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12, fontweight='bold')
# plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12, fontweight='bold')
# # plt.title('Figure 1: AI Performance Stress Test\nStratified by Radiologist Diagnostic Accuracy', 
# #           fontsize=14, fontweight='bold', pad=20)
# plt.grid(alpha=0.3)

# # ------------------------------------------------------------------
# # LAYOUT ADJUSTMENTS
# # ------------------------------------------------------------------

# # 1. Legend at TOP LEFT (Inside)
# plt.legend(loc='upper left', fontsize=9, framealpha=0.95, frameon=True)

# # 2. Performance Box at BOTTOM RIGHT (Inside)
# caption_text = (
#     f"Performance at Threshold = {TARGET_THRESHOLD:.3f}\n"
#     f"---------------------------------\n"
#     f"OVERALL COHORT:\n"
#     f"  • Sensitivity: {sens_all:.2%}\n"
#     f"  • Specificity: {spec_all:.2%}\n\n"
#     f"RAD CORRECT (Typical Cases):\n"
#     f"  • Sensitivity: {sens_corr:.2%}\n"
#     f"  • Specificity: {spec_corr:.2%}\n\n"
#     f"RAD INCORRECT (Stress Test):\n"
#     f"  • Sensitivity: {sens_inc:.2%}\n"
#     f"  • Specificity: {spec_inc:.2%}"
# )

# # Coordinates (0.55, 0.03) target the bottom right
# plt.text(0.75, 0.03, caption_text, fontsize=8.5,
#          bbox=dict(facecolor='white', alpha=0.95, edgecolor='lightgray', boxstyle='round,pad=0.5'))

# plt.tight_layout()
# plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from plotnine import * # pip install plotnine

# --- PRE-PROCESSING: Create a Long-Format DataFrame ---

# 1. Combine Curve Data
curves_data = []

# Helper to pack data
def pack_data(fpr, tpr, cohort_name, auc, low, high):
    temp_df = pd.DataFrame({'FPR': fpr, 'TPR': tpr})
    # Create the full label with AUC for the legend
    label = f"{cohort_name}\n(AUC={auc:.2f} [{low:.2f}-{high:.2f}])"
    temp_df['Cohort'] = label
    temp_df['LineType'] = cohort_name # simpler tag for styling
    return temp_df

# Pack the three cohorts
df_overall = pack_data(fpr_all, tpr_all, "Overall", auc_all, ci_low_all, ci_high_all)
df_corr = pack_data(fpr_corr, tpr_corr, "Radiologist-Correct", auc_corr, ci_low_corr, ci_high_corr)
df_inc = pack_data(fpr_inc, tpr_inc, "Radiologist-Error", auc_inc, ci_low_inc, ci_high_inc)

df_curves = pd.concat([df_overall, df_corr, df_inc])

# 2. Combine Point Data (The Operating Points)
points_data = pd.DataFrame({
    'FPR': [fpr_dot_all, fpr_dot_corr, fpr_dot_inc],
    'TPR': [tpr_dot_all, tpr_dot_corr, tpr_dot_inc],
    'Cohort': [df_overall['Cohort'].iloc[0], df_corr['Cohort'].iloc[0], df_inc['Cohort'].iloc[0]],
    'Shape': ['o', 's', '^'] # For mapping shapes
})

# 3. Prepare the Caption Text
caption_text = (
    f"Performance at Threshold = {TARGET_THRESHOLD:.3f}\n"
    f"---------------------------------\n"
    f"OVERALL COHORT:\n"
    f"  • Sensitivity: {sens_all:.2%}\n"
    f"  • Specificity: {spec_all:.2%}\n\n"
    f"RAD CORRECT (Typical Cases):\n"
    f"  • Sensitivity: {sens_corr:.2%}\n"
    f"  • Specificity: {spec_corr:.2%}\n\n"
    f"RAD INCORRECT (Stress Test):\n"
    f"  • Sensitivity: {sens_inc:.2%}\n"
    f"  • Specificity: {spec_inc:.2%}"
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Set the style
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 9))

# 1. Plot the Curves
sns.lineplot(
    data=df_curves, 
    x='FPR', 
    y='TPR', 
    hue='Cohort', 
    style='Cohort',
    palette=['navy', 'forestgreen', 'firebrick'],
    linewidth=2.5
)

# 2. Plot the Operating Points
colors = ['navy', 'forestgreen', 'firebrick']
markers = ['o', 's', '^']

for i, row in points_data.iterrows():
    plt.scatter(
        row['FPR'], row['TPR'], 
        color=colors[i], 
        marker=markers[i], 
        s=150, zorder=5, edgecolor='white', linewidth=1.5
    )

# 3. Plot Diagonal
plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle=':', zorder=1)

# 4. Customization
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12, fontweight='bold')
plt.legend(loc='upper left', fontsize=9, framealpha=0.95)

# --- 5. THE FIXED TEXT BOX ---

# Adjustments made:
# 1. Shortened the "───" line. Since you are not using monospace, 
#    you need fewer dashes to match the width of the text.
# 2. Reduced padding in bbox from 0.8 to 0.5.

caption_text = (
    f"Performance at Threshold = {TARGET_THRESHOLD:.3f}\n"
    f"─────────────────────────────\n" # Reduced length to match non-monospace text width
    f"Overall Cohort:\n"
    f"• Sensitivity: {sens_all:.2%}\n"
    f"• Specificity: {spec_all:.2%}\n\n"
    # Uncomment these if you need them, but ensure the line above isn't too short for them
    # f"Radiologist Correct:\n"
    # f"• Sensitivity: {sens_corr:.2%}\n"
    # f"• Specificity: {spec_corr:.2%}\n\n"
    # f"Radiologist Error (Stress Test):\n"
    # f"• Sensitivity: {sens_inc:.2%}\n"
    # f"• Specificity: {spec_inc:.2%}"
)

plt.text(
    0.96, 0.03,            
    caption_text, 
    fontsize=9, 
    ha='right',            
    va='bottom',           
    multialignment='left', 
    fontfamily='monospace', # Keep commented out if you prefer standard font
    bbox=dict(facecolor='white', alpha=0.95, edgecolor='lightgray', boxstyle='round,pad=0.5')
)

plt.tight_layout()
plt.show()

# Figure 2

In [ ]:
THRESHOLD = 0.536

df = result_df.copy()

# Define Order: Benign (TN, FP) -> Malignant (FN, TP)
order_list = ['TN', 'FP', 'FN', 'TP']

# ==========================================
# 2. Visualization Configuration
# ==========================================
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 8))

# Define a custom palette
# Greens for "Easy/Correct" (TN, TP), Oranges/Reds for "Hard/Incorrect" (FP, FN)
# Or distinct colors for all. Here we use a professional clinical palette.
palette_map = {
    'TN': '#2ca02c', # Green (Good)
    'FP': '#ff7f0e', # Orange (Warning)
    'FN': '#d62728', # Red (Danger)
    'TP': '#1f77b4'  # Blue (Detection)
}

# ==========================================
# 3. Create the Plot
# ==========================================

# A. The Violin Plot (Shows distribution density)
# inner='box' draws a mini boxplot inside the violin
ax = sns.violinplot(
    data=df, 
    x='Group', 
    y='ensemble_patient_score', 
    order=order_list,
    palette=palette_map,
    inner='box',       # Show quartiles inside
    linewidth=1.5,
    cut=0,             # Don't extend density past min/max data range
    alpha=0.3          # Transparency for the fill
)

# B. The Strip Plot (Shows individual data points)
# We jitter points to prevent overlap
sns.stripplot(
    data=df, 
    x='Group', 
    y='ensemble_patient_score',
    order=order_list,
    color='black',     # Black dots stand out against colors
    size=4, 
    alpha=0.4,         # Semi-transparent to show overlap
    jitter=0.2,
    ax=ax
)

# ==========================================
# 4. Annotations & Thresholds
# ==========================================

# A. The Threshold Line
plt.axhline(y=THRESHOLD, color='black', linestyle='--', linewidth=2, zorder=0)
plt.text(3.6, THRESHOLD + 0.02, f'Decision Threshold\n(t={THRESHOLD})', 
         fontsize=10, ha='left', va='bottom', color='black', fontweight='bold')

# B. Descriptive Titles
# plt.title('Figure 2: AI "Stress Test" Performance by Clinical Group', 
#           fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Clinical Subgroup (Ground Truth vs. Clinical Expectation)', fontsize=12, fontweight='bold')
plt.ylabel('AI Malignancy Probability Score', fontsize=12, fontweight='bold')

# C. Value-Add Annotations (The "Interpretation")
# These arrows highlight the clinical impact

# 1. Highlight "Avoided Biopsies" in FP group (Points below line)
# Note: Coordinates (x, y) depend on your data range. FP is index 1.
plt.annotate('Potential\nAvoided Biopsies', 
             xy=(1, 0.2), xytext=(1.5, 0.1),
             arrowprops=dict(facecolor='black', arrowstyle='->', lw=1.5),
             fontsize=10, ha='center', color='darkgreen', fontweight='bold')

# 2. Highlight "AI Rescue" in FN group (Points above line)
# Note: FN is index 2.
plt.annotate('AI Rescued\n(Occult Cancers)', 
             xy=(2, 0.8), xytext=(1.5, 0.9),
             arrowprops=dict(facecolor='black', arrowstyle='->', lw=1.5),
             fontsize=10, ha='center', color='darkblue', fontweight='bold')

# ==========================================
# 5. Final Formatting
# ==========================================
plt.ylim(-0.05, 1.05) # Keep strictly within 0-1 bounds with padding
plt.tight_layout()

plt.show()

## Table 2

In [ ]:
# ==========================================
# 1. Setup & Data Simulation 
# ==========================================
TARGET_THRESHOLD = 0.536

# Simulate the "Stress Test" cohorts
# FP Cohort (N=200): Difficult Benigns
df_stress = df[df['Group'].isin(['FP', 'FN'])].copy()

# ==========================================
# 2. Helper Function: Bootstrap CI
# ==========================================
def calculate_bootstrap_ci(scores, threshold, metric_type='sensitivity', n_bootstraps=2000, rng_seed=42):
    """
    Calculates 95% CI for Sensitivity or Specificity using Bootstrapping.
    
    Args:
        scores (array-like): Probability scores for the specific cohort.
        threshold (float): Decision threshold.
        metric_type (str): 'sensitivity' (TP rate) or 'specificity' (TN rate).
    """
    rng = np.random.RandomState(rng_seed)
    bootstrapped_metrics = []
    scores = np.array(scores)
    n = len(scores)
    
    for _ in range(n_bootstraps):
        # Resample with replacement
        indices = rng.randint(0, n, n)
        sample_scores = scores[indices]
        
        # Calculate metric on the sample
        if metric_type == 'specificity':
            # Specificity = TN / (TN + FP). In an all-negative cohort (FP group), this is just the accuracy.
            # Success = Score < Threshold
            metric = np.mean(sample_scores < threshold)
        elif metric_type == 'sensitivity':
            # Sensitivity = TP / (TP + FN). In an all-positive cohort (FN group), this is just the accuracy.
            # Success = Score >= Threshold
            metric = np.mean(sample_scores >= threshold)
            
        bootstrapped_metrics.append(metric)

    # Get percentiles for 95% CI
    lower = np.percentile(bootstrapped_metrics, 2.5)
    upper = np.percentile(bootstrapped_metrics, 97.5)
    
    return lower, upper

# ==========================================
# 3. Calculate "Stress Test" Metrics
# ==========================================

# --- Row 1: Avoiding Unnecessary Biopsies (Original FP Group) ---
fp_cohort = df_stress[df_stress['Group'] == 'FP']
fp_scores_arr = fp_cohort['ensemble_patient_score'].astype(float).values
total_fp = len(fp_cohort)

# Point Estimate (Specificity)
correct_fp = np.sum(fp_scores_arr < TARGET_THRESHOLD)
rate_fp = correct_fp / total_fp

# Bootstrap CI
low_fp, high_fp = calculate_bootstrap_ci(fp_scores_arr, TARGET_THRESHOLD, metric_type='specificity')

# --- Row 2: Detecting Occult Cancers (Original FN Group) ---
fn_cohort = df_stress[df_stress['Group'] == 'FN']
fn_scores_arr = fn_cohort['ensemble_patient_score'].astype(float).values
total_fn = len(fn_cohort)

# Point Estimate (Sensitivity)
correct_fn = np.sum(fn_scores_arr >= TARGET_THRESHOLD)
rate_fn = correct_fn / total_fn

# Bootstrap CI
low_fn, high_fn = calculate_bootstrap_ci(fn_scores_arr, TARGET_THRESHOLD, metric_type='sensitivity')

# ==========================================
# 4. Construct the Table
# ==========================================
table_data = [
    {
        "Clinical Challenge Scenario": "Avoiding Unnecessary\nBiopsies (Original\nFP Cohort)",
        "Target Metric": "Specificity",
        "Total Cases (N)": total_fp,
        "Correctly Classified\nby AI (n)": correct_fp,
        "Performance Rate\n(95% CI)": f"{rate_fp:.1%} ({low_fp:.1%}-{high_fp:.1%})",
        "Clinical Interpretation": f"Potential to reduce benign biopsies by\n{rate_fp:.1%} in this difficult group."
    },
    {
        "Clinical Challenge Scenario": "Detecting Occult\nCancers (Original\nFN Cohort)",
        "Target Metric": "Sensitivity",
        "Total Cases (N)": total_fn,
        "Correctly Classified\nby AI (n)": correct_fn,
        "Performance Rate\n(95% CI)": f"{rate_fn:.1%} ({low_fn:.1%}-{high_fn:.1%})",
        "Clinical Interpretation": f"Acts as a safety net, identifying {rate_fn:.1%}\nof cancers missed on initial mammography."
    }
]

df_table = pd.DataFrame(table_data)

# ==========================================
# 5. Render as Figure
# ==========================================
def render_mpl_table(data, col_width=3.5, row_height=0.625, font_size=12,
                     header_color='#2c3e50', row_colors=['#f1f1f2', 'w'], edge_color='w',
                     bbox=[0, 0, 1, 1], ax=None, **kwargs):
    if ax is None:
        size = (np.array(data.shape[::-1]) + np.array([0, 1])) * np.array([col_width, row_height])
        fig, ax = plt.subplots(figsize=size)
        ax.axis('off')

    mpl_table = ax.table(cellText=data.values, bbox=bbox, colLabels=data.columns, **kwargs)
    mpl_table.auto_set_font_size(False)
    mpl_table.set_fontsize(font_size)

    for k, cell in mpl_table._cells.items():
        cell.set_edgecolor(edge_color)
        if k[0] == 0:
            cell.set_text_props(weight='bold', color='w')
            cell.set_facecolor(header_color)
        else:
            cell.set_facecolor(row_colors[k[0]%len(row_colors) ])
            if k[1] == 0 or k[1] == 5: 
                cell.set_text_props(ha='left')
            else:
                cell.set_text_props(ha='center')
    return ax

display_df = df_table.copy()
display_df.columns = [
    "Clinical Challenge\nScenario", "Target\nMetric", "Total\nCases (N)", 
    "Correctly Classified\nby AI (n)", "Performance Rate\n(95% CI)", "Clinical Interpretation"
]

plt.figure(figsize=(14, 4))
ax = plt.gca()
render_mpl_table(display_df, header_color='#111111', row_colors=['#f8f9fa', '#ffffff'], ax=ax)
plt.title('Table 2: Primary "Stress Test" Outcomes – Clinical Utility in Challenge Cohorts', 
          loc='left', fontsize=14, fontweight='bold', pad=20)
plt.show()

# Table 3: Subgroup analysis

In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# 1. Setup & Data Transformation
# ==========================================
# df = result_df.copy() # Use your actual dataframe
TARGET_THRESHOLD = 0.536

def preprocess_for_table3(df):
    data = df.copy()
    
    # --- A. Density Transformation ---
    def map_density(val):
        val = str(val).upper().strip()
        if val in ['C', 'D']:
            return 'Dense (C/D)'
        elif val in ['A', 'B']:
            return 'Non-Dense (A/B)'
        return 'Unknown'
    
    data['Density_Strat'] = data['DensityCategory_std'].apply(map_density)
    
    # --- B. Lesion Morphology Transformation (4 GROUPS) ---
    # We now strictly separate Mixed from Asymmetry
    def map_lesion(val):
        val = str(val).lower()
        
        # 1. Mixed (Explicit check first)
        if 'mixed' in val:
            return 'Mixed Lesions'
        
        # 2. Asymmetry / Distortion
        elif 'distortion' in val or 'asymmetry' in val:
            return 'Asymmetry / Distortion'
            
        # 3. Calcification
        elif 'calcification' in val:
            return 'Calcification'
            
        # 4. Mass
        elif 'mass' in val or 'nodule' in val:
            return 'Mass / Nodule'
            
        else:
            return 'Other' 
            
    data['Lesion_Strat'] = data['LesionType_coarse'].apply(map_lesion)
    
    # --- C. Age Transformation ---
    data['PatientAge'] = data['PatientAge'].fillna(data['PatientAge'].mean())
    data['Age_Strat'] = data['PatientAge'].apply(lambda x: '< 50 Years' if x < 50 else '≥ 50 Years')
    
    return data

df_table = preprocess_for_table3(df)

# ==========================================
# 2. Isolate Cohorts
# ==========================================
fn_cohort = df_table[df_table['Group'] == 'FN'].copy()
fp_cohort = df_table[df_table['Group'] == 'FP'].copy()

# ==========================================
# 3. Calculation Logic
# ==========================================
def calculate_metrics(subgroup_col, subgroup_label):
    # --- RESCUE RATE (Sensitivity in FN) ---
    if subgroup_label == 'Overall':
        curr_fn = fn_cohort
    else:
        curr_fn = fn_cohort[fn_cohort[subgroup_col] == subgroup_label]
        
    n_fn = len(curr_fn)
    if n_fn > 0:
        rescued = (curr_fn['ensemble_patient_score'] >= TARGET_THRESHOLD).sum()
        rescue_pct = (rescued / n_fn) * 100
        rescue_str = f"{rescue_pct:.1f}% ({rescued}/{n_fn})"
    else:
        rescue_str = "N/A (0)"

    # --- AVOIDED BIOPSY RATE (Specificity in FP) ---
    if subgroup_label == 'Overall':
        curr_fp = fp_cohort
    else:
        curr_fp = fp_cohort[fp_cohort[subgroup_col] == subgroup_label]
        
    n_fp = len(curr_fp)
    if n_fp > 0:
        avoided = (curr_fp['ensemble_patient_score'] < TARGET_THRESHOLD).sum()
        avoided_pct = (avoided / n_fp) * 100
        avoided_str = f"{avoided_pct:.1f}% ({avoided}/{n_fp})"
    else:
        avoided_str = "N/A (0)"
        
    return {
        'Clinical Subgroup': subgroup_label,
        'AI Rescue Rate (FN Cohort)': rescue_str,
        'AI Avoided Biopsy Rate (FP Cohort)': avoided_str
    }

# ==========================================
# 4. Generate Rows
# ==========================================
rows = []

# 1. Overall
rows.append(calculate_metrics(None, 'Overall'))

# 2. Density
rows.append({'Clinical Subgroup': '--- Breast Density ---', 'AI Rescue Rate (FN Cohort)': '', 'AI Avoided Biopsy Rate (FP Cohort)': ''})
for cat in ['Non-Dense (A/B)', 'Dense (C/D)']:
    rows.append(calculate_metrics('Density_Strat', cat))

# 3. Lesion Morphology (Now 4 Distinct Groups)
rows.append({'Clinical Subgroup': '--- Lesion Morphology ---', 'AI Rescue Rate (FN Cohort)': '', 'AI Avoided Biopsy Rate (FP Cohort)': ''})
for cat in ['Mass / Nodule', 'Calcification', 'Mixed Lesions', 'Asymmetry / Distortion']:
    rows.append(calculate_metrics('Lesion_Strat', cat))

# 4. Age
rows.append({'Clinical Subgroup': '--- Patient Age ---', 'AI Rescue Rate (FN Cohort)': '', 'AI Avoided Biopsy Rate (FP Cohort)': ''})
for cat in ['< 50 Years', '≥ 50 Years']:
    rows.append(calculate_metrics('Age_Strat', cat))

# ==========================================
# 5. Output
# ==========================================
final_table = pd.DataFrame(rows)
print(final_table.to_markdown(index=False))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# ==========================================
# 1. Define Data Structure
# ==========================================
# Format: (Label, Rescue Rate, Avoided Biopsy Rate, Is_Section_Header)
data = [
    ("Overall", "25.0% (50/200)", "71.0% (142/200)", False),
    ("Breast Density", "", "", True),  # Section Header
    ("Non-Dense (A/B)", "27.5% (11/40)", "83.8% (31/37)", False),
    ("Dense (C/D)", "24.4% (39/160)", "68.1% (111/163)", False),
    ("Lesion Morphology", "", "", True), # Section Header
    ("Mass / Nodule", "28.9% (11/38)", "83.6% (107/128)", False),
    ("Calcification", "25.7% (18/70)", "25.9% (7/27)", False),
    ("Mixed Lesions", "33.3% (12/36)", "51.6% (16/31)", False),
    ("Asymmetry / Distortion", "N/A (0)", "85.7% (12/14)", False),
    ("Patient Age", "", "", True), # Section Header
    ("< 50 Years", "20.6% (22/107)", "72.2% (96/133)", False),
    ("≥ 50 Years", "30.1% (28/93)", "68.7% (46/67)", False)
]

columns = ["Clinical Subgroup", "AI Rescue Rate\n(Sensitivity in Occult Cancers)", "AI Avoided Biopsy Rate\n(Specificity in Benign Recalls)"]

# ==========================================
# 2. Configure Figure
# ==========================================
fig, ax = plt.subplots(figsize=(11, 8)) # Width, Height
ax.axis('off')

# ==========================================
# 3. Create & Style Table
# ==========================================
# Extract just the text for the table data
cell_text = [[row[0], row[1], row[2]] for row in data]

table = plt.table(cellText=cell_text,
                  colLabels=columns,
                  loc='center',
                  cellLoc='center',
                  bbox=[0, 0, 1, 1])

# --- Visual Styling Constants ---
HEADER_COLOR = '#2c3e50'   # Dark Blue-Grey
SECTION_COLOR = '#ecf0f1'  # Very Light Grey (for Density, Age headers)
TEXT_COLOR = 'black'
BORDER_COLOR = '#bdc3c7'

table.auto_set_font_size(False)
table.set_fontsize(11)

# Apply specific styles to cells
for (row, col), cell in table.get_celld().items():
    # Set default border
    cell.set_edgecolor(BORDER_COLOR)
    cell.set_linewidth(0.5)

    # A. Column Headers (Row 0)
    if row == 0:
        cell.set_text_props(weight='bold', color='white', fontsize=11)
        cell.set_facecolor(HEADER_COLOR)
        cell.set_height(0.08)
    
    # B. Data Rows (Row > 0)
    else:
        # Check if this row is a Section Header
        is_section = data[row-1][3]
        
        if is_section:
            cell.set_facecolor(SECTION_COLOR)
            cell.set_text_props(weight='bold', color='black')
            cell.set_height(0.05)
            # Left align section titles
            if col == 0:
                cell.set_text_props(ha='left')
                cell.get_text().set_position((0.02, 0.5)) # Slight indent
        else:
            cell.set_facecolor('white')
            cell.set_height(0.06)
            
            # Formatting the Clinical Subgroup Column (Col 0)
            if col == 0:
                cell.set_text_props(ha='left')
                # Indent regular items to look hierarchically below section headers
                if data[row-1][0] != "Overall": 
                    cell.get_text().set_position((0.05, 0.5)) 
                else:
                    cell.set_text_props(weight='bold') # Bold "Overall"
            
            # Formatting Number Columns (Col 1 & 2)
            else:
                # Dim the text slightly if it's "N/A"
                if "N/A" in data[row-1][col]:
                    cell.set_text_props(color='#7f8c8d') 

# ==========================================
# 4. Add Title & Export
# ==========================================
plt.title("Table 3: Stratified Diagnostic Utility in the 'Radiologist-Error' Cohort", 
          fontsize=14, fontweight='bold', y=1.02, x=0, loc='left')

# plt.savefig("Table3_HighRes.png", dpi=300, bbox_inches='tight') # Uncomment to save
plt.show()

# Logistic regression

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ==========================================
# 1. Load Data
# ==========================================
df = result_df.copy()

# ==========================================
# 2. Robust Analysis Function
# ==========================================
def run_robust_logistic(df, target_col, predictor_candidates, label_map):
    valid_predictors = []
    dropped_msgs = []

    # --- Step A: Validate Predictors ---
    for col in predictor_candidates:
        if col not in df.columns:
            dropped_msgs.append(f"❌ {col} (Missing)")
            continue
        
        # Check variance (ignoring NaNs)
        unique_vals = df[col].dropna().unique()
        if len(unique_vals) < 2:
            val = unique_vals[0] if len(unique_vals) > 0 else 'NaN'
            dropped_msgs.append(f"⚠️ {col} (Dropped: Constant value '{val}' in this cohort)")
            continue
            
        valid_predictors.append(col)

    # --- Step B: Construct Formula ---
    if not valid_predictors:
        print(f"   🔴 CRITICAL: No valid predictors left for target '{target_col}'")
        return "Error: No valid predictors"

    formula = f"{target_col} ~ {' + '.join(valid_predictors)}"
    
    # Print warnings
    for msg in dropped_msgs:
        print(f"   {msg}")

    # --- Step C: Run Model ---
    try:
        model = smf.logit(formula=formula, data=df).fit(disp=0)
        
        # Extract Results
        params = model.params
        conf = model.conf_int()
        conf['OR'] = params
        conf.columns = ['2.5%', '97.5%', 'OR']
        
        results = np.exp(conf)
        results['P-value'] = model.pvalues
        
        # Formatting
        summary = pd.DataFrame()
        summary['Odds Ratio (OR)'] = results['OR'].round(2)
        summary['95% CI'] = results.apply(lambda x: f"{x['2.5%']:.2f} – {x['97.5%']:.2f}", axis=1)
        summary['P-value'] = results['P-value'].round(3)
        
        summary = summary.drop('Intercept', errors='ignore')
        summary = summary.rename(index=label_map)
        
        return summary

    except Exception as e:
        return f"Error in Regression: {e}"

def safe_print_results(result, title):
    print(f"\n--- {title} ---")
    if isinstance(result, str):
        print(f"⚠️ ANALYSIS FAILED: {result}")
    else:
        print(result.to_string())

# ==========================================
# 3. Feature Engineering (Using LesionType_coarse)
# ==========================================

# 1. Fill NaNs
df['LesionType_coarse'] = df['LesionType_coarse'].fillna('Other/Unknown')
df['PatientAge'] = df['PatientAge'].fillna(df['PatientAge'].mean())

# 2. Age & Density
df['Age_50plus'] = (df['PatientAge'] >= 50).astype(int)
df['Is_Dense'] = df['DensityCategory_std'].isin(['C', 'D']).astype(int)

# 3. Lesion Types (Based on your provided distribution)
# Reference Group = "Mass/Nodule dominant" (so we DO NOT create a flag for it)

# Calcification dominant
df['Is_Calcification'] = df['LesionType_coarse'].str.contains('Calcification', case=False, na=False).astype(int)

# Architectural distortion/asymmetry
df['Is_Asymmetry'] = df['LesionType_coarse'].str.contains('asymmetry|distortion', case=False, na=False).astype(int)

# Mixed lesions (New predictor!)
df['Is_Mixed'] = df['LesionType_coarse'].str.contains('Mixed', case=False, na=False).astype(int)

# Labels for Output
labels_map = {
    'Is_Dense': 'Dense Breast (vs. Fatty)',
    'Is_Calcification': 'Lesion Type: Calcification (vs. Mass)',
    'Is_Asymmetry': 'Lesion Type: Asymmetry/Distortion (vs. Mass)',
    'Is_Mixed': 'Lesion Type: Mixed (vs. Mass)',
    'Age_50plus': 'Patient Age (≥50 vs. <50)'
}

# Predictor List (Including Mixed now)
predictors = ['Is_Dense', 'Is_Calcification', 'Is_Asymmetry', 'Is_Mixed', 'Age_50plus']

# ==========================================
# 4. Generate Table 4A: Predictors of Missed Cancers (FN)
# ==========================================
# Population: All Malignancies (TP + FN)
cancer_cohort = df[df['Group'].isin(['TP', 'FN'])].copy()
cancer_cohort['Outcome_Missed'] = (cancer_cohort['Group'] == 'FN').astype(int)

df_result_a = run_robust_logistic(cancer_cohort, 'Outcome_Missed', predictors, labels_map)
safe_print_results(df_result_a, "Table 4A: Predictors of Missed Cancers")

# ==========================================
# 5. Generate Table 4B: Predictors of False Alarms (FP)
# ==========================================
# Population: All Benigns (TN + FP)
benign_cohort = df[df['Group'].isin(['TN', 'FP'])].copy()
benign_cohort['Outcome_FalseAlarm'] = (benign_cohort['Group'] == 'FP').astype(int)

df_result_b = run_robust_logistic(benign_cohort, 'Outcome_FalseAlarm', predictors, labels_map)
safe_print_results(df_result_b, "Table 4B: Predictors of False Alarms")

In [ ]:
# cancer_cohort.head()

In [ ]:
# cancer_cohort['Is_Dense'].value_counts()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# 1. UPDATED DATA
# ==========================================

# Data for Table A (False Negatives - Missed Cancers)
# REMOVED Asymmetry because of Perfect Separation (OR=0.00)
data_a = [
    ["Dense Breast (vs. Fatty)", "1.03", "0.63 – 1.68", "0.899"],
    ["Lesion Type: Calcification (vs. Mass)", "2.69", "1.65 – 4.41", "<0.001"],
    ["Lesion Type: Mixed (vs. Mass)", "0.73", "0.45 – 1.19", "0.207"],
    ["Patient Age (≥50 vs. <50)", "0.55", "0.37 – 0.83", "0.004"]
]

# Data for Table B (False Positives - False Alarms)
# Asymmetry is kept here because it has valid numbers (0.75 OR)
data_b = [
    ["Dense Breast (vs. Fatty)", "0.67", "0.38 – 1.17", "0.163"],
    ["Lesion Type: Calcification (vs. Mass)", "0.69", "0.40 – 1.18", "0.179"],
    ["Lesion Type: Asymmetry/Distortion (vs. Mass)", "0.75", "0.36 – 1.53", "0.428"],
    ["Lesion Type: Mixed (vs. Mass)", "1.75", "0.94 – 3.26", "0.078"],
    ["Patient Age (≥50 vs. <50)", "1.61", "1.03 – 2.52", "0.037"]
]

columns = ["Predictor Variable", "Odds Ratio (OR)", "95% Confidence Interval", "P-value"]
df_a = pd.DataFrame(data_a, columns=columns)
df_b = pd.DataFrame(data_b, columns=columns)

# ==========================================
# 2. Table Rendering Function
# ==========================================

def render_logistic_table(df_a, df_b, title):
    # Adjusted figure size
    fig, ax = plt.subplots(figsize=(11, 10)) 
    ax.axis('off')
    
    header_color = '#2c3e50'
    subheader_color = '#95a5a6'
    row_colors = ['#f8f9fa', 'white']
    edge_color = 'white'
    
    # Rows: Header + Subheader A + Data A + Subheader B + Data B
    total_rows = 1 + 1 + len(df_a) + 1 + len(df_b)
    total_cols = len(columns)
    
    tbl = plt.table(cellText=[['']*total_cols]*total_rows, 
                    colLabels=columns, 
                    loc='center', cellLoc='center', bbox=[0, 0, 1, 1])
    
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(11)
    
    current_row = 0
    
    # Header
    for col in range(total_cols):
        cell = tbl[current_row, col]
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor(header_color)
        cell.set_edgecolor(edge_color)
        cell.set_height(0.08)
    current_row += 1
    
    # Subheader A
    tbl[current_row, 0].get_text().set_text("A. Predictors of Missed Cancers (False Negatives)")
    tbl[current_row, 0].set_text_props(weight='bold', color='white', ha='left')
    for col in range(total_cols):
        cell = tbl[current_row, col]
        cell.set_facecolor(subheader_color)
        cell.set_edgecolor(edge_color)
        cell.set_height(0.06)
    current_row += 1
    
    # Data A
    for i, row in df_a.iterrows():
        for col, val in enumerate(row):
            cell = tbl[current_row, col]
            cell.get_text().set_text(val)
            cell.set_facecolor(row_colors[i % 2])
            cell.set_edgecolor(edge_color)
            cell.set_height(0.06)
            if col == 0: cell.set_text_props(ha='left', fontfamily='sans-serif')
            
            # Bold Significant P-values
            if col == 3:
                try:
                    if '<' in str(val): cell.set_text_props(weight='bold')
                    elif float(val) < 0.05: cell.set_text_props(weight='bold')
                except: pass
        current_row += 1

    # Subheader B
    tbl[current_row, 0].get_text().set_text("B. Predictors of False Alarms (False Positives)")
    tbl[current_row, 0].set_text_props(weight='bold', color='white', ha='left')
    for col in range(total_cols):
        cell = tbl[current_row, col]
        cell.set_facecolor(subheader_color)
        cell.set_edgecolor(edge_color)
        cell.set_height(0.06)
    current_row += 1
    
    # Data B
    for i, row in df_b.iterrows():
        for col, val in enumerate(row):
            cell = tbl[current_row, col]
            cell.get_text().set_text(val)
            cell.set_facecolor(row_colors[i % 2])
            cell.set_edgecolor(edge_color)
            cell.set_height(0.06)
            if col == 0: cell.set_text_props(ha='left', fontfamily='sans-serif')
            
            # Bold Significant P-values
            if col == 3:
                try:
                    if '<' in str(val): cell.set_text_props(weight='bold')
                    elif float(val) < 0.05: cell.set_text_props(weight='bold')
                except: pass
        current_row += 1
        
    plt.title(title, fontsize=14, fontweight='bold', pad=20)
    plt.show()

render_logistic_table(df_a, df_b, "Table 4: Multivariate Logistic Regression Analysis of AI Errors")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ==========================================
# 1. Load Data
# ==========================================
df = result_df.copy()

# ==========================================
# 2. Robust Analysis Function
# ==========================================
def run_robust_logistic(df, target_col, predictor_candidates, label_map):
    valid_predictors = []
    dropped_msgs = []

    # --- Step A: Validate Predictors ---
    for col in predictor_candidates:
        if col not in df.columns:
            dropped_msgs.append(f"❌ {col} (Missing)")
            continue
        
        # Check variance (ignoring NaNs)
        unique_vals = df[col].dropna().unique()
        if len(unique_vals) < 2:
            val = unique_vals[0] if len(unique_vals) > 0 else 'NaN'
            dropped_msgs.append(f"⚠️ {col} (Dropped: Constant value '{val}' in this cohort)")
            continue
            
        valid_predictors.append(col)

    # --- Step B: Construct Formula ---
    if not valid_predictors:
        print(f"   🔴 CRITICAL: No valid predictors left for target '{target_col}'")
        return "Error: No valid predictors"

    formula = f"{target_col} ~ {' + '.join(valid_predictors)}"
    
    # Print warnings
    for msg in dropped_msgs:
        print(f"   {msg}")

    # --- Step C: Run Model ---
    try:
        model = smf.logit(formula=formula, data=df).fit(disp=0)
        
        # Extract Results
        params = model.params
        conf = model.conf_int()
        conf['OR'] = params
        conf.columns = ['2.5%', '97.5%', 'OR']
        
        results = np.exp(conf)
        results['P-value'] = model.pvalues
        
        # Formatting
        summary = pd.DataFrame()
        summary['Odds Ratio (OR)'] = results['OR'].round(2)
        summary['95% CI'] = results.apply(lambda x: f"{x['2.5%']:.2f} – {x['97.5%']:.2f}", axis=1)
        summary['P-value'] = results['P-value'].round(3)
        
        summary = summary.drop('Intercept', errors='ignore')
        summary = summary.rename(index=label_map)
        
        return summary

    except Exception as e:
        return f"Error in Regression: {e}"

def safe_print_results(result, title):
    print(f"\n--- {title} ---")
    if isinstance(result, str):
        print(f"⚠️ ANALYSIS FAILED: {result}")
    else:
        print(result.to_string())

# ==========================================
# 3. Feature Engineering (OPTION A: MERGE ASYMMETRY)
# ==========================================

# 1. Fill NaNs
df['LesionType_coarse'] = df['LesionType_coarse'].fillna('Other/Unknown')
df['PatientAge'] = df['PatientAge'].fillna(df['PatientAge'].mean())
# Handle missing density if necessary (using fillna('B') or similar as a safeguard)
# df['DensityCategory_std'] = df['DensityCategory_std'].fillna('B') 

# 2. Age & Density
df['Age_50plus'] = (df['PatientAge'] >= 50).astype(int)
df['Is_Dense'] = df['DensityCategory_std'].isin(['C', 'D']).astype(int)

# 3. Lesion Types
# Reference Group = "Mass/Nodule dominant"

# A. Calcification dominant
df['Is_Calcification'] = df['LesionType_coarse'].str.contains('Calcification', case=False, na=False).astype(int)

# B. Mixed + Asymmetry (MERGED)
# We combine Mixed, Asymmetry, and Distortion into one "Complex/Non-Mass" group
# to ensure we have enough 'missed cancer' events to calculate a valid Odds Ratio.
df['Is_Mixed_Asym'] = df['LesionType_coarse'].str.contains('Mixed|asymmetry|distortion', case=False, na=False).astype(int)

# Labels for Output
labels_map = {
    'Is_Dense': 'Dense Breast (vs. Fatty)',
    'Is_Calcification': 'Lesion Type: Calcification (vs. Mass)',
    'Is_Mixed_Asym': 'Lesion Type: Mixed/Asymmetry (vs. Mass)', # Updated Label
    'Age_50plus': 'Patient Age (≥50 vs. <50)'
}

# Predictor List (Removed 'Is_Asymmetry', Added 'Is_Mixed_Asym')
predictors = ['Is_Dense', 'Is_Calcification', 'Is_Mixed_Asym', 'Age_50plus']

# ==========================================
# 4. Generate Table 4A: Predictors of Missed Cancers (FN)
# ==========================================
# Population: All Malignancies (TP + FN)
cancer_cohort = df[df['Group'].isin(['TP', 'FN'])].copy()
cancer_cohort['Outcome_Missed'] = (cancer_cohort['Group'] == 'FN').astype(int)

df_result_a = run_robust_logistic(cancer_cohort, 'Outcome_Missed', predictors, labels_map)
safe_print_results(df_result_a, "Table 4A: Predictors of Missed Cancers")

# ==========================================
# 5. Generate Table 4B: Predictors of False Alarms (FP)
# ==========================================
# Population: All Benigns (TN + FP)
benign_cohort = df[df['Group'].isin(['TN', 'FP'])].copy()
benign_cohort['Outcome_FalseAlarm'] = (benign_cohort['Group'] == 'FP').astype(int)

df_result_b = run_robust_logistic(benign_cohort, 'Outcome_FalseAlarm', predictors, labels_map)
safe_print_results(df_result_b, "Table 4B: Predictors of False Alarms")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# 1. INPUT DATA (Based on your latest results)
# ==========================================

# Data for Table A (False Negatives - Missed Cancers)
data_a = [
    ["Dense Breast (vs. Fatty)", "1.03", "0.63 – 1.69", "0.892"],
    ["Lesion Type: Calcification (vs. Mass)", "2.70", "1.65 – 4.41", "<0.001"], # Formatted 0.000 -> <0.001
    ["Lesion Type: Mixed/Asymmetry (vs. Mass)", "0.67", "0.41 – 1.09", "0.105"],
    ["Patient Age (≥50 vs. <50)", "0.56", "0.37 – 0.84", "0.005"]
]

# Data for Table B (False Positives - False Alarms)
data_b = [
    ["Dense Breast (vs. Fatty)", "0.69", "0.39 – 1.20", "0.185"],
    ["Lesion Type: Calcification (vs. Mass)", "0.69", "0.40 – 1.18", "0.172"],
    ["Lesion Type: Mixed/Asymmetry (vs. Mass)", "1.22", "0.75 – 2.00", "0.421"],
    ["Patient Age (≥50 vs. <50)", "1.66", "1.06 – 2.59", "0.026"]
]

columns = ["Predictor Variable", "Odds Ratio (OR)", "95% Confidence Interval", "P-value"]

# Create DataFrames
df_a = pd.DataFrame(data_a, columns=columns)
df_b = pd.DataFrame(data_b, columns=columns)

# ==========================================
# 2. Table Rendering Function
# ==========================================

def render_logistic_table(df_a, df_b, title):
    # Figure setup
    fig, ax = plt.subplots(figsize=(11, 10))
    ax.axis('off')
    
    # --- Visual Settings ---
    header_color = '#2c3e50'    # Dark Slate Blue
    subheader_color = '#95a5a6' # Grayish Blue
    row_colors = ['#f8f9fa', 'white']
    edge_color = 'white'
    
    # Structure dimensions
    total_rows = 1 + 1 + len(df_a) + 1 + len(df_b)
    total_cols = len(columns)
    
    # Create the table object
    tbl = plt.table(cellText=[['']*total_cols]*total_rows, 
                    colLabels=columns, 
                    loc='center', 
                    cellLoc='center',
                    bbox=[0, 0, 1, 1])
    
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(11)
    
    # --- Formatting Loop ---
    
    current_row = 0
    
    # 1. Main Header
    for col in range(total_cols):
        cell = tbl[current_row, col]
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor(header_color)
        cell.set_edgecolor(edge_color)
        cell.set_height(0.08)
    current_row += 1
    
    # 2. Subheader A
    tbl[current_row, 0].get_text().set_text("A. Predictors of Missed Cancers (False Negatives)")
    tbl[current_row, 0].set_text_props(weight='bold', color='white', ha='left')
    for col in range(total_cols):
        cell = tbl[current_row, col]
        cell.set_facecolor(subheader_color)
        cell.set_edgecolor(edge_color)
        cell.set_height(0.06)
    current_row += 1
    
    # 3. Data A
    for i, row in df_a.iterrows():
        for col, val in enumerate(row):
            cell = tbl[current_row, col]
            cell.get_text().set_text(val)
            cell.set_facecolor(row_colors[i % 2])
            cell.set_edgecolor(edge_color)
            cell.set_height(0.06)
            
            # Left align Predictor Names
            if col == 0:
                cell.set_text_props(ha='left', fontfamily='sans-serif')
            
            # BOLD Significant P-values (< 0.05)
            if col == 3:
                try:
                    # Handle "<0.001" or numeric strings
                    if '<' in str(val) or float(val) < 0.05:
                        cell.set_text_props(weight='bold')
                except ValueError:
                    pass
        current_row += 1

    # 4. Subheader B
    tbl[current_row, 0].get_text().set_text("B. Predictors of False Alarms (False Positives)")
    tbl[current_row, 0].set_text_props(weight='bold', color='white', ha='left')
    for col in range(total_cols):
        cell = tbl[current_row, col]
        cell.set_facecolor(subheader_color)
        cell.set_edgecolor(edge_color)
        cell.set_height(0.06)
    current_row += 1
    
    # 5. Data B
    for i, row in df_b.iterrows():
        for col, val in enumerate(row):
            cell = tbl[current_row, col]
            cell.get_text().set_text(val)
            cell.set_facecolor(row_colors[i % 2])
            cell.set_edgecolor(edge_color)
            cell.set_height(0.06)
            
            # Left align Predictor Names
            if col == 0:
                cell.set_text_props(ha='left', fontfamily='sans-serif')
            
            # BOLD Significant P-values
            if col == 3:
                try:
                    if '<' in str(val) or float(val) < 0.05:
                        cell.set_text_props(weight='bold')
                except ValueError:
                    pass
        current_row += 1
        
    plt.title(title, fontsize=14, fontweight='bold', pad=20)
    plt.show()

# Render the final figure
render_logistic_table(df_a, df_b, "Table 4: Multivariate Logistic Regression Analysis of AI Errors")

In [ ]:
# Check the raw counts of Outcome vs. Lesion Type
cancer_cohort = df[df['Group'].isin(['TP', 'FN'])].copy()

print("--- Raw Counts: Asymmetry in Cancer Cohort ---")
# This will show you exactly how many TPs vs FNs exist for Asymmetry
cross_tab = pd.crosstab(cancer_cohort['LesionType_coarse'], cancer_cohort['Group'])
print(cross_tab)